In [4]:
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from rdkit import Chem

# Load the saved GAN-generated vectors
df_generated = pd.read_csv("generated_molecules_vectors.csv")
generated_vectors = df_generated.to_numpy()

# Load Chemformer in Python 3.7.11 environment
model_path = "/home/jovyan/Research Project/sdf_data/Chemformer/molbart/inference_score.py"
chemformer = AutoModelForSeq2SeqLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

def vector_to_smiles(vector):
    """
    Converts a latent vector into a SMILES string using Chemformer.
    """
    with torch.no_grad():
        input_tokens = torch.tensor(vector).unsqueeze(0)
        output_tokens = chemformer.generate(input_tokens)
        smiles = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    return smiles if Chem.MolFromSmiles(smiles) else None  # Validate molecule

# Convert GAN-generated vectors to SMILES using Chemformer
generated_smiles = [vector_to_smiles(vec) for vec in generated_vectors]

# Filter out invalid molecules
generated_smiles = [smi for smi in generated_smiles if smi]

# Save decoded SMILES to a file
df_smiles = pd.DataFrame(generated_smiles, columns=["SMILES"])
df_smiles.to_csv("decoded_molecules_smiles.csv", index=False)

print("✅ Successfully converted vectors to SMILES and saved to 'decoded_molecules_smiles.csv'.")


OSError: Incorrect path_or_model_id: 'Chemformer/molbart/inference_score.py'. Please provide either the path to a local folder or the repo_id of a model on the Hub.